# Experiment 79: Pseudo Label 품질 고도화 — 다중 모델 합의 + 신뢰도 가중

**목표**: LB 10.12838 돌파

**핵심 가설**:
```
현재 PL 현황:
  - 70b 예측(LB 10.129) 기반 PL 사용 중
  - 이후 72_top2(10.1287), 75_g5(10.1284)로 더 좋은 모델이 나왔음
  - PL Round 3에서 수렴 → 하지만 이건 70b 품질에서의 수렴
  - 더 좋은 PL → 같은 GBDT라도 성능 향상

개선 전략 3가지:
  ① 다중 모델 합의 PL:
     72_top2, 75_g5, 77_full_only 등 최선 제출들의 가중 평균
     → 개별 모델의 noise 상쇄 → 더 정확한 PL

  ② Confidence-weighted PL:
     여러 모델 예측의 분산(disagreement) 기반 연속 가중치
     → 기존 전략C의 이진 필터(포함/제외)보다 세밀
     → 분산 작은 행: 높은 weight (모델들이 동의)
     → 분산 큰 행: 낮은 weight (불확실)

  ③ Iterative refinement:
     새 PL → GBDT 재학습 → 새 예측 → 또 PL 갱신
     → 70b 기반이 아닌 새 출발점에서 수렴
```

**실행 구조**:
```
Stage 0: 기존 제출 파일 분석 (합의 PL 구성)                [~5분]
Stage 1: Round 1 — 합의 PL로 GBDT 10-seed 학습           [~2시간]
Stage 2: Round 1 결과로 PL 갱신 → Round 2 학습            [~2시간]
Stage 3: 최종 블렌딩 + 제출 파일 생성                      [~5분]
```

In [ ]:
# Cell 1: 패키지 설치 및 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

!pip install -q lightgbm xgboost catboost

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print("설치 완료")

In [ ]:
# Cell 2: Import 및 경로 설정
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from scipy.stats import pearsonr
import os, sys, warnings, gc

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
warnings.filterwarnings('ignore')

DATA_DIR    = '/content/drive/MyDrive/MyDrive/LightGBM'
TRAIN_PATH  = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH   = os.path.join(DATA_DIR, 'test.csv')
LAYOUT_PATH = os.path.join(DATA_DIR, 'layout_info.csv')

# ── 기존 제출 파일들 (PL 합의에 사용) ──
# 사용 가능한 모든 제출 파일을 여기에 나열
SUBMISSION_FILES = {
    '70b':      os.path.join(DATA_DIR, 'submission_70b.csv'),         # LB 10.129
    '72_top2':  os.path.join(DATA_DIR, 'submission_72_top2.csv'),     # LB 10.1287
    '75_g5':    os.path.join(DATA_DIR, 'submission_75_g5.csv'),       # LB 10.1284
    '77_full':  os.path.join(DATA_DIR, 'submission_77_full_only.csv'),# LB 10.1299
}

TARGET = 'avg_delay_minutes_next_30m'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

In [ ]:
# Cell 3: 유틸 함수
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if   c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:  df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
    return df

In [ ]:
# Cell 4: preprocess_data (Exp 67 원본 — 수정 금지)
def preprocess_data(train, test, layout):
    print("Preprocessing...")
    train = train.merge(layout, on='layout_id', how='left')
    test  = test.merge(layout,  on='layout_id', how='left')

    le = LabelEncoder()
    le.fit(pd.concat([train['layout_type'], test['layout_type']]).astype(str))
    train['layout_type'] = le.transform(train['layout_type'].astype(str))
    test['layout_type']  = le.transform(test['layout_type'].astype(str))

    for df in [train, test]:
        df['robot_utilization']    = df['robot_active']       / (df['robot_total']          + 1e-6)
        df['charger_utilization']  = df['robot_charging']     / (df['charger_count']         + 1e-6)
        df['aisle_pressure']       = df['congestion_score']   / (df['aisle_width_avg']        + 1e-6)
        df['intersection_density'] = df['intersection_count'] / (df['floor_area_sqm']         + 1e-6)
        df['pack_station_pressure']= df['order_inflow_15m']   / (df['pack_station_count']     + 1e-6)
        df['bottleneck_risk']      = df['congestion_score'] * df['intersection_density'] / (df['aisle_width_avg'] + 1e-6)
        df['task_intensity']       = df['order_inflow_15m']   / (df['robot_active']           + 1e-6)

    layout_num_cols = ['aisle_width_avg', 'intersection_count', 'robot_total']
    key_ops         = ['congestion_score', 'robot_active', 'order_inflow_15m']
    for l_col in layout_num_cols:
        for o_col in key_ops:
            train[f'{l_col}_x_{o_col}'] = train[l_col] * train[o_col]
            test[f'{l_col}_x_{o_col}']  = test[l_col]  * test[o_col]

    for col in ['congestion_score', 'low_battery_ratio', 'robot_active']:
        train[f'{col}_vel'] = train.groupby('scenario_id')[col].diff(1)
        test[f'{col}_vel']  = test.groupby('scenario_id')[col].diff(1)

    train['time_idx'] = train.groupby('scenario_id').cumcount()
    test['time_idx']  = test.groupby('scenario_id').cumcount()
    train = train.sort_values(['scenario_id', 'time_idx'])
    test  = test.sort_values(['scenario_id', 'time_idx'])

    SEQ_COLS = [
        'order_inflow_15m','unique_sku_15m','robot_active','robot_idle',
        'robot_charging','battery_mean','battery_std','low_battery_ratio',
        'charge_queue_length','avg_charge_wait','congestion_score',
        'max_zone_density','blocked_path_15m','near_collision_15m',
        'fault_count_15m','avg_recovery_time','task_reassign_15m',
        'replenishment_overlap','pack_utilization','loading_dock_util',
        'staging_area_util','label_print_queue'
    ]

    for col in SEQ_COLS:
        first_tr = train.groupby('scenario_id')[col].transform('first')
        train[f'{col}_vs_start']    = train[col] / (first_tr + 1e-6)
        train[f'{col}_delta_start'] = train[col] - first_tr
        first_ts = test.groupby('scenario_id')[col].transform('first')
        test[f'{col}_vs_start']    = test[col] / (first_ts + 1e-6)
        test[f'{col}_delta_start'] = test[col] - first_ts

    for col in SEQ_COLS:
        if col not in train.columns: continue
        for df in [train, test]:
            prev    = df.groupby('scenario_id')[col].shift(1)
            cum_max = prev.groupby(df['scenario_id']).cummax()
            cum_min = prev.groupby(df['scenario_id']).cummin()
            df[f'{col}_vs_cummax'] = df[col] / (cum_max + 1e-6)
            df[f'{col}_vs_cummin'] = df[col] / (cum_min.abs() + 1e-6)
            cum_range = cum_max - cum_min
            df[f'{col}_position_in_range'] = ((df[col] - cum_min) / (cum_range + 1e-3)).clip(0, 2)

    lag_cols = [
        'congestion_score','fault_count_15m','charge_queue_length',
        'low_battery_ratio','blocked_path_15m','avg_recovery_time',
        'near_collision_15m','pack_utilization'
    ]
    for col in lag_cols:
        if col not in train.columns: continue
        for df in [train, test]:
            for lag in [4,5,6,7]: df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [7,10,14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    SEQ_COLS_BASE = ['order_inflow_15m','unique_sku_15m','robot_active',
                     'low_battery_ratio','charge_queue_length','congestion_score','fault_count_15m']
    remaining = [c for c in SEQ_COLS_BASE if c not in lag_cols]
    for col in remaining:
        for df in [train, test]:
            for lag in [4,5]: df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [7,14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    for col in SEQ_COLS:
        for df in [train, test]:
            for lag in [8,10,12,15,20,24]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)

    remaining2 = [c for c in SEQ_COLS if c not in lag_cols]
    for col in remaining2:
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    tr_new = train['scenario_id'].values != np.roll(train['scenario_id'].values, 1); tr_new[0] = True
    ts_new = test['scenario_id'].values  != np.roll(test['scenario_id'].values,  1); ts_new[0]  = True
    for col in SEQ_COLS_BASE:
        for lag in [1,2]:
            tr_l = train[col].shift(lag).values.copy(); ts_l = test[col].shift(lag).values.copy()
            for l in range(lag):
                tr_l[np.roll(tr_new, l)] = np.nan; ts_l[np.roll(ts_new, l)] = np.nan
            train[f'{col}_lag{lag}'] = tr_l; test[f'{col}_lag{lag}'] = ts_l
        train[f'{col}_exp_mean'] = train.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())
        test[f'{col}_exp_mean']  = test.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())

    train['time_ratio'] = train.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    test['time_ratio']  = test.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    for df in [train, test]:
        df['congestion_ratio'] = df['congestion_score'] / (df['congestion_score_exp_mean'] + 1e-6)
        df['steps_remaining']  = df.groupby('scenario_id')['time_idx'].transform('max') - df['time_idx']

    train.fillna(0, inplace=True); test.fillna(0, inplace=True)
    return reduce_mem_usage(train), reduce_mem_usage(test)

In [ ]:
# Cell 5: apply_smoothed_te
def apply_smoothed_te(df_tr, df_val, target_col, k=30):
    global_mean = df_tr[target_col].mean()
    agg = df_tr.groupby('layout_id')[target_col].agg(['mean','std','median','count']).reset_index()
    agg['layout_mean'] = (agg['count'] * agg['mean'] + k * global_mean) / (agg['count'] + k)
    agg.rename(columns={'std':'layout_std','median':'layout_median','count':'layout_count'}, inplace=True)
    df_val = df_val.merge(agg[['layout_id','layout_mean','layout_std','layout_median','layout_count']], on='layout_id', how='left')
    df_tr  = df_tr.merge( agg[['layout_id','layout_mean','layout_std','layout_median','layout_count']], on='layout_id', how='left')
    df_val['layout_mean']   = df_val['layout_mean'].fillna(global_mean)
    df_val['layout_std']    = df_val['layout_std'].fillna(df_tr[target_col].std())
    df_val['layout_median'] = df_val['layout_median'].fillna(df_tr[target_col].median())
    df_val['layout_count']  = df_val['layout_count'].fillna(0)
    return df_tr, df_val, ['layout_mean','layout_std','layout_median','layout_count']

In [ ]:
# Cell 6: 공통 설정
zero_imp_features = [
    'charge_queue_length','avg_charge_wait','charge_queue_length_lag2','fault_count_15m_lag2',
    'time_ratio','task_reassign_15m_vs_cummin','low_battery_ratio_vel','low_battery_ratio_lag2',
    'task_reassign_15m','blocked_path_15m_lag8','blocked_path_15m_lag10','near_collision_15m_lag8',
    'near_collision_15m_lag10','fault_count_15m_lag8','fault_count_15m_lag10','avg_recovery_time_lag8',
    'avg_recovery_time_lag10','task_reassign_15m_lag8','task_reassign_15m_lag10','replenishment_overlap_lag8',
    'replenishment_overlap_lag10','robot_charging_lag10','low_battery_ratio_lag8','low_battery_ratio_lag10',
    'charge_queue_length_lag8','charge_queue_length_lag10','avg_charge_wait_lag8','avg_charge_wait_lag10',
    'fault_count_15m_vs_cummax','fault_count_15m_vs_cummin','avg_recovery_time_vs_cummax','task_reassign_15m_vs_cummax',
    'low_battery_ratio_vs_cummax','low_battery_ratio_vs_cummin','charge_queue_length_vs_cummin','avg_charge_wait_vs_cummax',
    'blocked_path_15m_vs_cummax','near_collision_15m_vs_cummin','charge_queue_length_roll14_mean','task_reassign_15m_vs_start',
    'avg_recovery_time_lag5','avg_recovery_time_lag6','avg_recovery_time_lag7','near_collision_15m_lag4',
    'near_collision_15m_lag5','near_collision_15m_lag6','near_collision_15m_lag7','charge_queue_length_roll7_std',
    'charge_queue_length_roll10_mean','low_battery_ratio_lag4','low_battery_ratio_lag5','low_battery_ratio_lag7',
    'blocked_path_15m_lag4','blocked_path_15m_lag5','blocked_path_15m_lag6','blocked_path_15m_lag7',
    'label_print_queue_delta_start','robot_charging_lag15','low_battery_ratio_lag12','low_battery_ratio_lag15',
    'charge_queue_length_lag12','charge_queue_length_lag15','avg_charge_wait_lag12','avg_charge_wait_lag15',
    'congestion_score_lag12','max_zone_density_lag15','blocked_path_15m_lag12','fault_count_15m_lag4',
    'fault_count_15m_lag5','fault_count_15m_lag6','fault_count_15m_lag7','charge_queue_length_lag4',
    'charge_queue_length_lag5','charge_queue_length_lag6','charge_queue_length_lag7','charge_queue_length_roll7_mean',
    'blocked_path_15m_lag15','near_collision_15m_lag12','near_collision_15m_lag15','fault_count_15m_lag12',
    'fault_count_15m_lag15','avg_recovery_time_lag12','avg_recovery_time_lag15','task_reassign_15m_lag12',
    'task_reassign_15m_lag15','replenishment_overlap_lag12','replenishment_overlap_lag15','charge_queue_length_position_in_range',
    'avg_charge_wait_position_in_range','congestion_score_position_in_range','blocked_path_15m_position_in_range',
    'near_collision_15m_position_in_range','fault_count_15m_position_in_range','avg_recovery_time_position_in_range',
    'task_reassign_15m_position_in_range','replenishment_overlap_position_in_range','label_print_queue_position_in_range',
    'replenishment_overlap_vs_cummin','staging_area_util_vs_cummax','battery_mean_position_in_range',
    'low_battery_ratio_position_in_range','label_print_queue_lag15','charge_queue_length_roll20_std',
    'charge_queue_length_vs_start','charge_queue_length_delta_start','avg_charge_wait_vs_start','avg_charge_wait_delta_start'
]

gkf   = GroupKFold(n_splits=5)
seeds = [42, 123, 2026, 777, 1004, 314, 555, 888, 999, 1337]

def target_forward(y): return np.log1p(y)
def target_inverse(p): return np.expm1(p)

# GBDT 파라미터 (Exp 67 확정값)
LGB_PARAMS = {
    'objective': 'huber', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 127, 'max_depth': -1,
    'min_child_samples': 20, 'subsample': 0.75, 'subsample_freq': 1,
    'colsample_bytree': 0.65, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_estimators': 3000, 'verbose': -1,
}
XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 7, 'subsample': 0.75,
    'colsample_bytree': 0.65, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_estimators': 3000, 'device': 'cuda', 'tree_method': 'hist', 'verbosity': 0,
}
CAT_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'learning_rate': 0.03, 'depth': 7, 'iterations': 3000,
    'l2_leaf_reg': 3.0, 'random_strength': 1.0, 'bagging_temperature': 0.8,
    'task_type': 'GPU', 'devices': '0', 'verbose': 0,
}

print("공통 설정 완료")

In [ ]:
# Cell 7: 데이터 로드 + 전처리
print("--- Experiment 79: Pseudo Label 품질 고도화 ---")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
layout    = pd.read_csv(LAYOUT_PATH)

train_layouts  = set(train_raw['layout_id'].unique())
test_layouts   = set(test_raw['layout_id'].unique())
seen_layouts   = train_layouts & test_layouts
unseen_layouts = test_layouts - train_layouts

train, test = preprocess_data(train_raw, test_raw, layout)

EXCLUDE_COLS    = ['ID','layout_id','scenario_id',TARGET,'is_seen','is_pseudo',
                   'pseudo_label','pseudo_std','pseudo_weight']
features_base   = [c for c in train.columns if c not in EXCLUDE_COLS]
features_pruned = [f for f in features_base if f not in zero_imp_features]

train['is_seen'] = train['layout_id'].isin(seen_layouts)
test['is_seen']  = test['layout_id'].isin(seen_layouts)

print(f"피처: {len(features_pruned)}개")
print(f"Train: {len(train)}행")
print(f"Test: {len(test)}행 (Seen {test['is_seen'].sum()}, Unseen {(~test['is_seen']).sum()})")

In [ ]:
# Cell 8: [Stage 0] 다중 모델 합의 PL 구성
#
# 핵심 아이디어:
#   1) 여러 제출 파일의 가중 평균 → 개별 noise 상쇄
#   2) 모델 간 예측 분산(std) → 신뢰도 가중치
#      - std 작은 행: 모델들이 동의 → 높은 weight
#      - std 큰 행: 모델들이 불일치 → 낮은 weight
#   3) 기존 전략C의 이진 필터도 유지 (극단값/저분산 layout)

print("[Stage 0] 다중 모델 합의 Pseudo Label 구성")
print("="*60)

# ── 제출 파일 로드 ──
preds = {}
for name, path in SUBMISSION_FILES.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        preds[name] = df.set_index('ID').loc[test['ID'].values][TARGET].values
        print(f"  ✅ {name}: 로드 완료 (mean={preds[name].mean():.4f})")
    else:
        print(f"  ❌ {name}: 파일 없음 — 스킵")

n_models = len(preds)
print(f"\n  사용 가능한 모델: {n_models}개")

if n_models < 2:
    print("  ⚠️ 모델이 2개 미만. 합의 PL 효과 제한적.")
    print("  → 가장 좋은 단일 모델을 PL로 사용합니다.")

# ── 모델 간 상관관계 ──
pred_names = list(preds.keys())
print(f"\n  [모델 간 상관관계]")
for i in range(len(pred_names)):
    for j in range(i+1, len(pred_names)):
        r, _ = pearsonr(preds[pred_names[i]], preds[pred_names[j]])
        print(f"    {pred_names[i]} ↔ {pred_names[j]}: r={r:.4f}")

# ── LB 기반 가중 평균 (Inverse-LB weighting) ──
# LB가 낮을수록 좋은 모델 → 높은 가중치
LB_SCORES = {
    '70b':     10.129,
    '72_top2': 10.1287,
    '75_g5':   10.1284,
    '77_full': 10.1299,
}

# 사용 가능한 모델만 필터
available_lbs = {k: v for k, v in LB_SCORES.items() if k in preds}
inv_lb = {k: 1.0 / v for k, v in available_lbs.items()}
total_inv = sum(inv_lb.values())
weights = {k: v / total_inv for k, v in inv_lb.items()}

print(f"\n  [LB 기반 가중치]")
for k, w in weights.items():
    print(f"    {k}: LB={available_lbs[k]:.4f} → weight={w:.4f}")

# ── 합의 PL 생성 ──
consensus_pl = np.zeros(len(test))
for name, w in weights.items():
    consensus_pl += w * preds[name]

# ── 모델 간 분산 (신뢰도 지표) ──
pred_matrix = np.column_stack([preds[name] for name in preds])
pred_std = pred_matrix.std(axis=1)    # 행별 표준편차
pred_mean_abs_dev = np.abs(pred_matrix - pred_matrix.mean(axis=1, keepdims=True)).mean(axis=1)

test['pseudo_label'] = consensus_pl
test['pseudo_std']   = pred_std

print(f"\n  [합의 PL 통계]")
print(f"    mean: {consensus_pl.mean():.4f}")
print(f"    std:  {consensus_pl.std():.4f}")
print(f"  [모델 간 분산 (pred_std)]")
print(f"    mean: {pred_std.mean():.4f}")
print(f"    median: {np.median(pred_std):.4f}")
print(f"    p95: {np.percentile(pred_std, 95):.4f}")
print(f"    max: {pred_std.max():.4f}")

# Seen vs Unseen 분산 비교
seen_mask = test['is_seen'].values
print(f"\n  [Seen vs Unseen 분산]")
print(f"    Seen   mean_std: {pred_std[seen_mask].mean():.4f}")
print(f"    Unseen mean_std: {pred_std[~seen_mask].mean():.4f}")

print("\n[Stage 0 완료]")

In [ ]:
# Cell 9: Confidence-weighted Pseudo 데이터 구성 함수
#
# [기존 전략C vs 신규 전략D 비교]
#
# 전략C (기존):
#   - 이진 필터: 극단값 제외, 저분산 Unseen layout 제외
#   - 포함된 행은 모두 동일 weight (Seen=1.0, Unseen=0.2)
#
# 전략D (신규):
#   - 전략C의 이진 필터 유지 (안전장치)
#   - 추가로 모델 간 분산 기반 연속 가중치
#   - 분산 작은 행(모델 동의) → weight 1.0
#   - 분산 큰 행(모델 불일치) → weight 0.2~0.5
#   - → 기존보다 세밀한 가중치 부여

def build_pseudo_dataset(train, test, consensus_col='pseudo_label',
                         std_col='pseudo_std', strategy='D'):
    """
    Pseudo Label 데이터셋 구성
    
    strategy:
      'C' — 기존 이진 필터 (Exp 70b와 동일)
      'D' — 이진 필터 + confidence weighting (신규)
    
    Returns:
      train_combined: 원본 train + pseudo test
      pseudo_stats: 통계 정보
    """
    train_std = train[TARGET].std()
    train_q01 = train[TARGET].quantile(0.01)
    train_q99 = train[TARGET].quantile(0.99)
    
    # ── 이진 필터 (전략C, D 공통) ──
    unseen_test    = test[~test['is_seen']]
    layout_std_map = unseen_test.groupby('layout_id')[consensus_col].std()
    low_var_lay    = layout_std_map[layout_std_map < train_std * 0.3].index.tolist()
    mask_low_var   = test['layout_id'].isin(low_var_lay) & ~test['is_seen']
    mask_extreme   = (test[consensus_col] < train_q01) | (test[consensus_col] > train_q99)
    
    pseudo_mask = (
        (test['is_seen'] & ~mask_extreme) |
        (~test['is_seen'] & ~mask_low_var & ~mask_extreme)
    )
    
    pseudo_set = test[pseudo_mask].copy()
    pseudo_set[TARGET]      = pseudo_set[consensus_col]
    pseudo_set['is_pseudo'] = True
    
    # ── Confidence weight 계산 (전략D만) ──
    if strategy == 'D' and std_col in test.columns:
        # 분산 기반 신뢰도: std가 작을수록 높은 weight
        pseudo_std_vals = pseudo_set[std_col].values
        
        # 정규화: [0, 1] → [w_min, 1.0]
        # p90 기준으로 정규화 (극단적 outlier에 강건)
        std_p90 = np.percentile(pseudo_std_vals, 90)
        std_normalized = np.clip(pseudo_std_vals / (std_p90 + 1e-6), 0, 2)
        
        # confidence: 1 - normalized_std → [0, 1]
        # w_min=0.2로 바운드 (너무 낮으면 pseudo 효과 소멸)
        w_min = 0.2
        confidence = np.clip(1.0 - std_normalized, 0, 1)
        pseudo_weight = w_min + (1.0 - w_min) * confidence
        
        pseudo_set['pseudo_weight'] = pseudo_weight
    else:
        pseudo_set['pseudo_weight'] = 1.0  # 전략C: 동일 weight
    
    # ── 결합 ──
    shared_cols = [c for c in train.columns if c in pseudo_set.columns]
    train_c = train.copy()
    train_c['is_pseudo'] = False
    train_c['pseudo_weight'] = 1.0  # 원본은 항상 weight=1.0
    
    if 'pseudo_weight' not in shared_cols:
        shared_cols.append('pseudo_weight')
    
    train_combined = pd.concat(
        [train_c[shared_cols], pseudo_set[shared_cols]], ignore_index=True)
    train_combined['is_pseudo'] = train_combined['is_pseudo'].fillna(False)
    
    # ── 통계 ──
    n_pseudo = pseudo_mask.sum()
    n_pseudo_seen   = (pseudo_mask & test['is_seen']).sum()
    n_pseudo_unseen = (pseudo_mask & ~test['is_seen']).sum()
    n_excluded = len(test) - n_pseudo
    
    stats = {
        'n_pseudo': n_pseudo,
        'n_pseudo_seen': n_pseudo_seen,
        'n_pseudo_unseen': n_pseudo_unseen,
        'n_excluded': n_excluded,
        'n_low_var': mask_low_var.sum(),
        'n_extreme': mask_extreme.sum(),
        'low_var_layouts': low_var_lay,
    }
    
    if strategy == 'D':
        stats['weight_mean'] = pseudo_set['pseudo_weight'].mean()
        stats['weight_median'] = pseudo_set['pseudo_weight'].median()
        stats['weight_min'] = pseudo_set['pseudo_weight'].min()
    
    return train_combined, stats


def make_sample_weight_v2(df, w_orig=1.0, w_seen_base=1.0, w_unseen_base=0.2):
    """
    전략D용 sample weight:
      원본 train: w_orig
      Pseudo Seen:   w_seen_base × pseudo_weight (confidence)
      Pseudo Unseen: w_unseen_base × pseudo_weight (confidence)
    """
    is_pseudo = df['is_pseudo'].fillna(False).values
    is_seen   = df['is_seen'].values
    pw        = df['pseudo_weight'].fillna(1.0).values
    
    return np.where(
        ~is_pseudo, w_orig,
        np.where(is_seen, w_seen_base * pw, w_unseen_base * pw)
    )


print("Pseudo 구성 함수 정의 완료")

In [ ]:
# Cell 10: 전략 C와 D 비교 + train_combined 구성

print("[전략 C vs D 비교]")
print("="*60)

# 전략 C (기존 — 비교 기준)
tc_c, stats_c = build_pseudo_dataset(train, test, strategy='C')
print(f"\n  전략 C (기존 이진 필터):")
print(f"    Pseudo 포함: {stats_c['n_pseudo']}행 (Seen {stats_c['n_pseudo_seen']}, Unseen {stats_c['n_pseudo_unseen']})")
print(f"    제외: {stats_c['n_excluded']}행 (저분산 {stats_c['n_low_var']}, 극단값 {stats_c['n_extreme']})")
print(f"    저분산 layout: {stats_c['low_var_layouts']}")

# 전략 D (신규 — confidence weighting)
tc_d, stats_d = build_pseudo_dataset(train, test, strategy='D')
print(f"\n  전략 D (이진 필터 + confidence weight):")
print(f"    Pseudo 포함: {stats_d['n_pseudo']}행 (동일 필터)")
print(f"    Weight 분포: mean={stats_d['weight_mean']:.4f}, median={stats_d['weight_median']:.4f}, min={stats_d['weight_min']:.4f}")

# 전략 D 사용 (합의 PL + confidence weight)
train_combined = tc_d
del tc_c  # 메모리 절약
gc.collect()

print(f"\n  ✅ 전략 D 채택: 합의 PL + confidence weighting")
print(f"  통합 train: {len(train_combined)}행 (원본 {len(train)} + pseudo {stats_d['n_pseudo']})")

In [ ]:
# Cell 11: GBDT 학습 함수 (전략D weight 적용)

def train_gbdt_round(train_combined, test, features, seeds, round_name="R1"):
    """
    GBDT 3종 × N-seed 앙상블 (전략D weight 적용)
    """
    orig_mask = ~train_combined['is_pseudo'].fillna(False).values
    y_orig    = train_combined[TARGET].values[orig_mask]
    n_orig    = orig_mask.sum()
    n_test    = len(test)
    
    oof  = {k: np.zeros(n_orig) for k in ['lgb','xgb','cat']}
    tprd = {k: np.zeros(n_test) for k in ['lgb','xgb','cat']}
    
    n_seeds = len(seeds)
    total_models = n_seeds * 5 * 3
    model_count = 0
    
    for seed_idx, seed in enumerate(seeds):
        print(f"\n{'='*25} {round_name} Seed {seed} ({seed_idx+1}/{n_seeds}) {'='*25}")
        
        oof_seed  = {k: np.zeros(len(train_combined)) for k in ['lgb','xgb','cat']}
        test_seed = {k: np.zeros(n_test) for k in ['lgb','xgb','cat']}
        
        for fold, (tr_idx, val_idx) in enumerate(
                gkf.split(train_combined, train_combined[TARGET],
                          groups=train_combined['layout_id'])):
            
            X_tr_raw  = train_combined.iloc[tr_idx].copy()
            y_tr_raw  = train_combined.iloc[tr_idx][TARGET].values
            sw_tr     = make_sample_weight_v2(train_combined.iloc[tr_idx])
            X_val_raw = train_combined.iloc[val_idx].copy()
            y_val_raw = train_combined.iloc[val_idx][TARGET].values
            
            # CV-safe TE
            X_tr_raw, X_val_raw, te_cols = apply_smoothed_te(
                X_tr_raw, X_val_raw, TARGET, k=30)
            _, X_te_raw, _ = apply_smoothed_te(
                X_tr_raw, test.copy(), TARGET, k=30)
            
            X_tr_raw.fillna(0, inplace=True)
            X_val_raw.fillna(0, inplace=True)
            X_te_raw.fillna(0, inplace=True)
            
            feats = features + te_cols
            y_tr_t  = target_forward(y_tr_raw)
            y_val_t = target_forward(y_val_raw)
            
            # ── LightGBM ──
            lgb_p = {**LGB_PARAMS, 'random_state': seed}
            m_lgb = LGBMRegressor(**lgb_p)
            m_lgb.fit(
                X_tr_raw[feats], y_tr_t, sample_weight=sw_tr,
                eval_set=[(X_val_raw[feats], y_val_t)],
                callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
            )
            p_val_lgb  = np.maximum(target_inverse(m_lgb.predict(X_val_raw[feats])), 0)
            p_test_lgb = np.maximum(target_inverse(m_lgb.predict(X_te_raw[feats])), 0)
            oof_seed['lgb'][val_idx]  = p_val_lgb
            test_seed['lgb']         += p_test_lgb / 5
            model_count += 1
            
            # ── XGBoost ──
            xgb_p = {**XGB_PARAMS, 'random_state': seed}
            m_xgb = xgb.XGBRegressor(**xgb_p)
            m_xgb.fit(
                X_tr_raw[feats], y_tr_t, sample_weight=sw_tr,
                eval_set=[(X_val_raw[feats], y_val_t)], verbose=False
            )
            p_val_xgb  = np.maximum(target_inverse(m_xgb.predict(X_val_raw[feats])), 0)
            p_test_xgb = np.maximum(target_inverse(m_xgb.predict(X_te_raw[feats])), 0)
            oof_seed['xgb'][val_idx]  = p_val_xgb
            test_seed['xgb']         += p_test_xgb / 5
            model_count += 1
            
            # ── CatBoost ──
            cat_p = {**CAT_PARAMS, 'random_seed': seed}
            m_cat = CatBoostRegressor(**cat_p)
            m_cat.fit(
                X_tr_raw[feats], y_tr_t, sample_weight=sw_tr,
                eval_set=(X_val_raw[feats], y_val_t),
                early_stopping_rounds=100, verbose=0
            )
            p_val_cat  = np.maximum(target_inverse(m_cat.predict(X_val_raw[feats])), 0)
            p_test_cat = np.maximum(target_inverse(m_cat.predict(X_te_raw[feats])), 0)
            oof_seed['cat'][val_idx]  = p_val_cat
            test_seed['cat']         += p_test_cat / 5
            model_count += 1
            
            fold_mae_lgb = mean_absolute_error(y_val_raw, p_val_lgb)
            fold_mae_xgb = mean_absolute_error(y_val_raw, p_val_xgb)
            fold_mae_cat = mean_absolute_error(y_val_raw, p_val_cat)
            print(f"  F{fold+1} LGB={fold_mae_lgb:.4f} XGB={fold_mae_xgb:.4f} CAT={fold_mae_cat:.4f}"
                  f"  [{model_count}/{total_models}]", flush=True)
            
            del m_lgb, m_xgb, m_cat; gc.collect()
        
        orig_indices = np.where(orig_mask)[0]
        for k in ['lgb','xgb','cat']:
            oof[k]  += oof_seed[k][orig_indices] / n_seeds
            tprd[k] += test_seed[k] / n_seeds
        
        seed_ens = (oof_seed['lgb'][orig_mask] + oof_seed['xgb'][orig_mask] + oof_seed['cat'][orig_mask]) / 3
        print(f"  Seed {seed} Ens OOF: {mean_absolute_error(y_orig, seed_ens):.4f}")
    
    # ── Inverse MAE 가중 앙상블 ──
    mae_lgb = mean_absolute_error(y_orig, oof['lgb'])
    mae_xgb = mean_absolute_error(y_orig, oof['xgb'])
    mae_cat = mean_absolute_error(y_orig, oof['cat'])
    inv_mae = np.array([1/mae_lgb, 1/mae_xgb, 1/mae_cat])
    weights = inv_mae / inv_mae.sum()
    
    oof_ens  = weights[0]*oof['lgb']  + weights[1]*oof['xgb']  + weights[2]*oof['cat']
    test_ens = weights[0]*tprd['lgb'] + weights[1]*tprd['xgb'] + weights[2]*tprd['cat']
    ens_mae  = mean_absolute_error(y_orig, oof_ens)
    
    print(f"\n  {round_name} LGB={mae_lgb:.4f} XGB={mae_xgb:.4f} CAT={mae_cat:.4f}")
    print(f"  {round_name} weights: LGB={weights[0]:.4f} XGB={weights[1]:.4f} CAT={weights[2]:.4f}")
    print(f"  {round_name} Ensemble OOF MAE: {ens_mae:.4f}")
    
    return {
        'test_pred': np.maximum(test_ens, 0),
        'oof_pred': oof_ens,
        'oof_mae': ens_mae,
        'weights': weights,
        'test_preds_individual': {'lgb': tprd['lgb'], 'xgb': tprd['xgb'], 'cat': tprd['cat']},
    }

print("GBDT 학습 함수 정의 완료")

In [ ]:
# Cell 12: [Stage 1] Round 1 — 합의 PL + Confidence Weight로 GBDT 학습
#
# 변경점 (기존 70b PL 대비):
#   1) PL 값: 70b 단독 → 다중 모델 가중 평균 (noise 감소)
#   2) Weight: 이진(1.0/0.2) → confidence 기반 연속 (세밀)
# 파이프라인 나머지는 Exp 67/70b와 100% 동일
#
# 소요 시간: 약 2시간

print("\n" + "="*70)
print(f"  [Stage 1] Round 1 — 합의 PL + Confidence Weight")
print(f"  PL: {n_models}개 모델 LB-weighted 평균")
print(f"  Weight: 전략D (confidence)")
print(f"  피처: {len(features_pruned)}개 + TE 4개")
print(f"  Train: {len(train_combined)}행")
print("="*70)

result_r1 = train_gbdt_round(
    train_combined, test, features_pruned, seeds, round_name="R1")

print(f"\n[Stage 1 완료] R1 OOF MAE: {result_r1['oof_mae']:.4f}")
print(f"  기존 Exp 70b OOF: 8.6935")
print(f"  기존 Exp 77 Full OOF: 8.6886")
print(f"  차이 (R1 - 77Full): {result_r1['oof_mae'] - 8.6886:+.4f}")

In [ ]:
# Cell 13: Round 1 제출 파일 저장 + 기존 best와 블렌딩

print("[Round 1 제출 파일 생성]")
print("="*60)

# 기존 best 로드
pred_72 = pd.read_csv(SUBMISSION_FILES['72_top2']).set_index('ID').loc[test['ID'].values][TARGET].values

try:
    pred_75 = pd.read_csv(SUBMISSION_FILES['75_g5']).set_index('ID').loc[test['ID'].values][TARGET].values
    has_75 = True
except:
    has_75 = False

saved_files = []

# ── R1 단독 ──
fname = 'submission_79_r1.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: result_r1['test_pred']}).to_csv(fpath, index=False)
saved_files.append(('R1 단독', fname))
print(f"  ✅ {fname} — Round 1 단독")

# ── R1 + 72_top2 블렌딩 ──
for alpha in [0.3, 0.5, 0.7]:
    blend = alpha * result_r1['test_pred'] + (1 - alpha) * pred_72
    blend = np.maximum(blend, 0)
    fname = f'submission_79_r1_b{int(alpha*10)}.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    saved_files.append((f'R1×{int(alpha*100)}% + 72×{int((1-alpha)*100)}%', fname))
    print(f"  ✅ {fname} — R1 {int(alpha*100)}% + 72_top2 {int((1-alpha)*100)}%")

# ── R1 + 75_g5 블렌딩 ──
if has_75:
    for alpha in [0.3, 0.5, 0.7]:
        blend = alpha * result_r1['test_pred'] + (1 - alpha) * pred_75
        blend = np.maximum(blend, 0)
        fname = f'submission_79_r1_75b{int(alpha*10)}.csv'
        fpath = os.path.join(DATA_DIR, fname)
        pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
        saved_files.append((f'R1×{int(alpha*100)}% + 75×{int((1-alpha)*100)}%', fname))
        print(f"  ✅ {fname} — R1 {int(alpha*100)}% + 75_g5 {int((1-alpha)*100)}%")

# ── 72_top2 대비 상관관계 ──
corr_r1_72, _ = pearsonr(result_r1['test_pred'], pred_72)
print(f"\n  R1 ↔ 72_top2 상관관계: {corr_r1_72:.4f}")
print(f"  R1 ↔ 72_top2 mean diff: {np.abs(result_r1['test_pred'] - pred_72).mean():.4f}")

print(f"\n총 {len(saved_files)}개 파일 저장")
print(f"\n📋 제출 순서:")
print(f"  1순위: submission_79_r1.csv (R1 단독 — 합의PL 효과 순수 측정)")
print(f"  2순위: submission_79_r1_b5.csv (R1 50% + 72_top2 50% — 안전)")
if has_75:
    print(f"  3순위: submission_79_r1_75b5.csv (R1 50% + 75_g5 50%)")

In [ ]:
# Cell 14: [Stage 2] Round 2 — R1 예측으로 PL 갱신 후 재학습
#
# R1의 test 예측을 새 PL로 사용 → 합의 PL을 한단계 더 개선
# R1 예측 + 기존 합의 PL을 다시 평균하여 더 정제된 PL 생성
#
# ⚠️ Stage 1 LB 결과를 확인한 후에 실행하는 것을 권장
# R1 LB가 기존보다 좋으면 → R2 진행
# R1 LB가 기존보다 나쁘면 → R2 스킵, 블렌딩만 탐색
#
# 소요 시간: 약 2시간

print("\n" + "="*70)
print(f"  [Stage 2] Round 2 — R1 예측으로 PL 갱신")
print("="*70)

# ── R1 예측을 기존 합의에 추가 ──
# 새 합의 = (기존 합의 × 0.5) + (R1 예측 × 0.5)
# R1은 합의 PL로 학습된 것이므로 합의 PL보다 더 좋을 가능성
consensus_r2 = 0.5 * consensus_pl + 0.5 * result_r1['test_pred']

# 분산 재계산 (R1 예측도 포함)
pred_matrix_r2 = np.column_stack(
    [preds[name] for name in preds] + [result_r1['test_pred']]
)
pred_std_r2 = pred_matrix_r2.std(axis=1)

test['pseudo_label'] = consensus_r2
test['pseudo_std']   = pred_std_r2

print(f"  R2 PL = 50% 합의(R0) + 50% R1 예측")
print(f"  R2 PL mean: {consensus_r2.mean():.4f}")
print(f"  R2 신뢰도(std) mean: {pred_std_r2.mean():.4f} (R1: {pred_std.mean():.4f})")

# 새 pseudo 데이터셋 구성
train_combined_r2, stats_r2 = build_pseudo_dataset(train, test, strategy='D')

print(f"  R2 pseudo: {stats_r2['n_pseudo']}행")
print(f"  R2 weight: mean={stats_r2['weight_mean']:.4f}")

# ── Round 2 학습 ──
result_r2 = train_gbdt_round(
    train_combined_r2, test, features_pruned, seeds, round_name="R2")

print(f"\n[Stage 2 완료] R2 OOF MAE: {result_r2['oof_mae']:.4f}")
print(f"  R1 OOF: {result_r1['oof_mae']:.4f}")
print(f"  차이 (R2 - R1): {result_r2['oof_mae'] - result_r1['oof_mae']:+.4f}")

In [ ]:
# Cell 15: [Stage 3] 최종 블렌딩 + 제출 파일 생성

print("[Stage 3] 최종 블렌딩")
print("="*60)

# ── R2 단독 ──
fname = 'submission_79_r2.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: result_r2['test_pred']}).to_csv(fpath, index=False)
print(f"  ✅ {fname} — R2 단독")

# ── R1 + R2 평균 (self-ensemble) ──
r1r2_avg = (result_r1['test_pred'] + result_r2['test_pred']) / 2
r1r2_avg = np.maximum(r1r2_avg, 0)
fname = 'submission_79_r1r2_avg.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: r1r2_avg}).to_csv(fpath, index=False)
print(f"  ✅ {fname} — R1+R2 평균 (self-ensemble)")

# ── R2 + 기존 best 블렌딩 ──
for alpha in [0.3, 0.5, 0.7]:
    blend = alpha * result_r2['test_pred'] + (1 - alpha) * pred_72
    blend = np.maximum(blend, 0)
    fname = f'submission_79_r2_b{int(alpha*10)}.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    print(f"  ✅ {fname} — R2 {int(alpha*100)}% + 72_top2 {int((1-alpha)*100)}%")

# ── R1R2 평균 + 기존 best 블렌딩 ──
for alpha in [0.3, 0.5, 0.7]:
    blend = alpha * r1r2_avg + (1 - alpha) * pred_72
    blend = np.maximum(blend, 0)
    fname = f'submission_79_r1r2_b{int(alpha*10)}.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    print(f"  ✅ {fname} — R1R2avg {int(alpha*100)}% + 72_top2 {int((1-alpha)*100)}%")

# ── 75_g5와 블렌딩 ──
if has_75:
    blend = 0.5 * result_r2['test_pred'] + 0.5 * pred_75
    blend = np.maximum(blend, 0)
    fname = 'submission_79_r2_75b5.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    print(f"  ✅ {fname} — R2 50% + 75_g5 50%")

# ── R1, R2, 72, 75 전체 평균 (mega ensemble) ──
mega_preds = [result_r1['test_pred'], result_r2['test_pred'], pred_72]
if has_75:
    mega_preds.append(pred_75)
mega_avg = np.maximum(np.mean(mega_preds, axis=0), 0)
fname = 'submission_79_mega_avg.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: mega_avg}).to_csv(fpath, index=False)
print(f"  ✅ {fname} — Mega ensemble 균등 평균")

print("\n전체 제출 파일 생성 완료")

In [ ]:
# Cell 16: 최종 결과 요약

print("\n" + "="*70)
print("  [Exp 79: Pseudo Label 품질 고도화 — 최종 결과]")
print("="*70)

# R1 ↔ R2 상관관계
corr_r1_r2, _ = pearsonr(result_r1['test_pred'], result_r2['test_pred'])
corr_r2_72, _ = pearsonr(result_r2['test_pred'], pred_72)

print(f"\n  {'모델':<35} {'OOF MAE':>10}")
print("-"*50)
print(f"  {'Exp 70b (기존 PL 기반)':>35} {'8.6935':>10}")
print(f"  {'Exp 77 Full (70b PL 재현)':>35} {'8.6886':>10}")
print(f"  {'R1 (합의 PL + confidence)':>35} {result_r1['oof_mae']:>10.4f}")
print(f"  {'R2 (R1 PL 갱신 후 재학습)':>35} {result_r2['oof_mae']:>10.4f}")

print(f"\n  상관관계:")
print(f"    R1 ↔ 72_top2: {corr_r1_72:.4f}")
print(f"    R2 ↔ 72_top2: {corr_r2_72:.4f}")
print(f"    R1 ↔ R2:      {corr_r1_r2:.4f}")

print(f"\n  기존 최선: LB 10.12838 (Exp 75)")

print(f"\n  {'─'*55}")
print(f"  📋 제출 순서:")
print(f"  {'─'*55}")
print(f"  1순위: submission_79_r1.csv")
print(f"         (합의 PL 효과 순수 측정)")
print(f"  2순위: submission_79_r1_b5.csv")
print(f"         (R1 50% + 72_top2 50% — 안전 블렌딩)")

if result_r2['oof_mae'] <= result_r1['oof_mae']:
    print(f"  3순위: submission_79_r2.csv")
    print(f"         (R2 OOF가 R1보다 좋으므로 기대)")
    print(f"  4순위: submission_79_r1r2_avg.csv")
    print(f"         (R1+R2 self-ensemble)")
else:
    print(f"  3순위: submission_79_r1r2_avg.csv")
    print(f"         (R2가 R1보다 나쁨 → 평균으로 안전)")
    print(f"  4순위: submission_79_mega_avg.csv")
    print(f"         (전체 모델 평균 — 최대 다양성)")

print(f"\n  🔍 판단 기준:")
print(f"  ┌───────────────────────────────────────────────────────────┐")
print(f"  │ R1 LB < 10.128                                         │")
print(f"  │   → 합의 PL + confidence weight 효과 확인!             │")
print(f"  │   → R2 진행, 이후 R3까지 iterative refinement          │")
print(f"  │   → 최종 예측 + TabNet 블렌딩으로 추가 개선            │")
print(f"  │                                                         │")
print(f"  │ R1 LB ≈ 10.128                                         │")
print(f"  │   → PL 품질 차이 미미. 블렌딩(b5)에서 미세 개선 가능   │")
print(f"  │   → R1+72+75 mega ensemble 탐색                        │")
print(f"  │                                                         │")
print(f"  │ R1 LB > 10.129                                         │")
print(f"  │   → 합의 PL이 오히려 noise 추가. 전략 전환 필요        │")
print(f"  │   → post-processing 탐색 또는 피처 재설계              │")
print(f"  └───────────────────────────────────────────────────────────┘")
print("="*70)